<a href="https://colab.research.google.com/github/kevinketerlondon-tech/Data-Analyst-portfolio/blob/main/00c_reconstruct_original35_pairs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 00c: Reconstruct the 35 before/after pairs of the earlier tile-based analysis
**Kevin Keter · University of East London**

The earlier change detection grouped images by ~500 m tile and compared the first image of the earliest year with
the first image of the latest year. This repeats that logic so the pairs can be checked in Notebook 02.
Inputs in the Drive folder: `manifest_v2.csv` and `dissertation_change_results_recovered.csv`. Output: `original35_pairs.csv`.

## 1. Settings

In [ ]:
import os, glob, zipfile
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/Dissertation/revision"
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True)
class Args: pass
args = Args()
args.manifest = f"{DRIVE}/manifest_v2.csv"
args.dissertation_results = f"{DRIVE}/dissertation_change_results_recovered.csv"
# If the file already exists (it was used for the reported results), write a second copy and compare,
# so re-running this notebook never overwrites the original.
MAIN = f"{DRIVE}/original35_pairs.csv"
args.out = MAIN if not os.path.exists(MAIN) else MAIN.replace(".csv", "_rerun.csv")
print("Output:", args.out)

Mounted at /content/drive
Output: /content/drive/MyDrive/Dissertation/revision/original35_pairs_rerun.csv


## 2. Run

In [ ]:
import pandas as pd

m = pd.read_csv(args.manifest, dtype={"image_id": str})
m = m[m.status.isin(["downloaded", "already_existed"])]
rows = []
for (area, tile), g in m.groupby(["area", "cluster_id"]):
    years = sorted(g.year.unique())
    if len(years) < 2: continue
    e, l = years[0], years[-1]
    rows.append(dict(area=area, cluster_id=tile, earliest_year=e, earliest_image=g[g.year == e].iloc[0].image_id,
                     latest_year=l, latest_image=g[g.year == l].iloc[0].image_id))
p = pd.DataFrame(rows)
r = pd.read_csv(args.dissertation_results)
x = p.merge(r, on=["area", "cluster_id"], suffixes=("", "_reported"))
assert len(x) == len(p) == 35 and (x.earliest_year == x.earliest_year_reported).all() and (x.latest_year == x.latest_year_reported).all()
x.drop(columns=["earliest_year_reported", "latest_year_reported"]).to_csv(args.out, index=False)
print(f"Reconstructed {len(x)} pairs; years match the dissertation output for all of them.")

Reconstructed 35 pairs; years match the dissertation output for all of them.


### Check against the file used in the paper

In [ ]:
if args.out != MAIN:
    a, b = pd.read_csv(MAIN, dtype=str), pd.read_csv(args.out, dtype=str)
    print("Identical to the pairs used in the paper:", a.equals(b))

Identical to the pairs used in the paper: True
